# PATH Step 2: Component Network Model with Step-1 Prognostic Risk Score

This notebook implements **Step 2** of the PATH-style analysis for the collaborative-care component network.

## Analysis definition

- **Outcome:** PHQ-9 measured within the prespecified approximately 12-month window:
  $$
  11.5 \leq t \leq 12.5.
  $$

- **Prognostic score:** the patient-level `risk_score` supplied by the Step-1 model.

- **Treatment representation:** 13 harmonized collaborative-care component indicators.

- **Effect modification:** each component is interacted with the Step-1 prognostic score.

- **Study adjustment:** study-specific intercepts are included.

The fitted mean structure is

$$
PHQ9_{12m,i}
=
\alpha_{\mathrm{study}(i)}
+
\lambda R_i
+
\sum_{k=1}^{K}\beta_k C_{ik}
+
\sum_{k=1}^{K}\delta_k C_{ik}R_i
+
\epsilon_i.
$$

where:

- $R_i$ is the Step-1 prognostic score for patient $i$;
- $C_{ik}$ indicates whether component $k$ is present in the patient's randomized intervention package;
- $\beta_k$ is the component main effect;
- $\delta_k$ is the component-by-risk interaction;
- $\alpha_{\mathrm{study}(i)}$ is a study-specific intercept.

This is the PATH-style extension from **treatment + risk + treatment × risk** to **components + risk + components × risk**.

**Current decision:** 
1. This notebook involves only 6 components 
2. This notebook involves only 7 datasets that are left after all the filtering steps (GAD-7 at baseline was among the conditions included) and for whom the risk scores are available



## 1. Setup

The cohort-construction logic remains in `risk_score_dataset.py`.

The Step-1 risk scores are expected at:

`../data/processed/risk_scores.csv`

when this notebook is run from the repository's `notebooks/` directory.


In [ ]:
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm

from risk_score_dataset import create


RISK_SCORES_PATH = Path("../data/processed/risk_scores.csv")

if not RISK_SCORES_PATH.exists():
    raise FileNotFoundError(
        f"Could not find Step-1 risk scores at: {RISK_SCORES_PATH.resolve()}"
    )

filtered_df = create()


## 2. Inspect the filtered longitudinal cohort

Check the number of eligible patients and studies after the agreed filtering rules.


In [ ]:
n_filtered_patients = (
    filtered_df[["STUDY_ID", "patient_id"]]
    .drop_duplicates()
    .shape[0]
)

n_filtered_studies = filtered_df["STUDY_ID"].nunique()

print("Unique patients:", n_filtered_patients)
print("Studies:", n_filtered_studies)


In [ ]:
study_summary = (
    filtered_df[["STUDY_ID", "patient_id"]]
    .drop_duplicates()
    .groupby("STUDY_ID")
    .size()
    .rename("n_patients")
    .sort_values(ascending=False)
)

study_summary


### Coventry 2015 cohort check

Coventry retains very few patients after filtering, so inspect its retained observations explicitly.


In [ ]:
filtered_df.loc[
    filtered_df["STUDY_ID"] == "08_Coventry_2015",
    [
        "STUDY_ID",
        "patient_id",
        "follow_up_months",
        "phq9_total",
        "age",
        "sex",
        "study_arm",
        "gad7_total",
    ],
].sort_values(["patient_id", "follow_up_months"])


## 3. Construct baseline and approximately 12-month observations

Baseline is defined as

$$
t = 0.
$$

The outcome is PHQ-9 observed within

$$
11.5 \leq t \leq 12.5.
$$

If a patient has more than one observation in the follow-up window, retain the observation closest to exactly 12 months.


In [ ]:
baseline = (
    filtered_df.loc[
        filtered_df["follow_up_months"] == 0.0
    ]
    .copy()
)


In [ ]:
followup = (
    filtered_df.loc[
        filtered_df["follow_up_months"].between(
            11.5,
            12.5,
            inclusive="both",
        )
    ]
    .copy()
)

followup["distance_from_12"] = (
    followup["follow_up_months"] - 12.0
).abs()

followup = (
    followup
    .sort_values(
        [
            "STUDY_ID",
            "patient_id",
            "distance_from_12",
        ]
    )
    .drop_duplicates(
        subset=["STUDY_ID", "patient_id"],
        keep="first",
    )
)

print("12-month outcome observations:", len(followup))
print(
    "Studies with 12-month outcomes:",
    followup["STUDY_ID"].nunique(),
)


In [ ]:
followup["follow_up_months"].value_counts().sort_index()


## 4. Build one patient-level analysis row

Create one row per patient containing:

- study and patient identifiers;
- randomized study arm;
- baseline characteristics retained by the Step-1 cohort;
- baseline PHQ-9;
- approximately 12-month PHQ-9;
- the actual month used for the outcome.

Baseline PHQ-9 remains available for description, but it is **not used as the Step-2 risk score** in this version.


In [ ]:
analysis_df = baseline[
    [
        "STUDY_ID",
        "patient_id",
        "study_arm",
        "age",
        "sex",
        "gad7_total",
        "phq9_total",
    ]
].copy()

analysis_df = analysis_df.rename(
    columns={
        "phq9_total": "phq9_baseline",
    }
)


In [ ]:
analysis_df = analysis_df.merge(
    followup[
        [
            "STUDY_ID",
            "patient_id",
            "follow_up_months",
            "phq9_total",
        ]
    ].rename(
        columns={
            "follow_up_months": "outcome_month",
            "phq9_total": "phq9_12m",
        }
    ),
    on=["STUDY_ID", "patient_id"],
    how="inner",
    validate="one_to_one",
)

print("Patients with baseline + approximately 12-month outcome:", len(analysis_df))


## 5. Load and merge the Step-1 prognostic risk score

The Step-1 file must contain:

- `STUDY_ID`
- `patient_id`
- `risk_score`

The risk score is merged by the composite patient identifier

$$
(\mathrm{STUDY\_ID}, \mathrm{patient\_id}).
$$

A **left merge** is used first so that patients without a Step-1 risk score can be identified rather than silently discarded.


In [ ]:
risk_scores = pd.read_csv(RISK_SCORES_PATH)

required_risk_columns = {
    "STUDY_ID",
    "patient_id",
    "risk_score",
}

missing_columns = required_risk_columns.difference(risk_scores.columns)

if missing_columns:
    raise ValueError(
        f"Risk-score file is missing required columns: {sorted(missing_columns)}"
    )

print("Risk-score rows:", len(risk_scores))
print(
    "Unique risk-score patients:",
    risk_scores[["STUDY_ID", "patient_id"]]
    .drop_duplicates()
    .shape[0],
)
print("Risk-score studies:", risk_scores["STUDY_ID"].nunique())
print("Missing risk scores:", risk_scores["risk_score"].isna().sum())


In [ ]:
duplicate_risk_ids = risk_scores.duplicated(
    subset=["STUDY_ID", "patient_id"],
    keep=False,
)

if duplicate_risk_ids.any():
    display(
        risk_scores.loc[
            duplicate_risk_ids,
            ["STUDY_ID", "patient_id", "risk_score"],
        ].sort_values(["STUDY_ID", "patient_id"])
    )
    raise ValueError(
        "Duplicate STUDY_ID + patient_id pairs found in risk_scores.csv"
    )

risk_scores["risk_score"].describe()


In [ ]:
analysis_df = analysis_df.merge(
    risk_scores[
        [
            "STUDY_ID",
            "patient_id",
            "risk_score",
        ]
    ],
    on=["STUDY_ID", "patient_id"],
    how="left",
    validate="one_to_one",
)

missing_risk_mask = analysis_df["risk_score"].isna()

print("Analysis patients before requiring risk score:", len(analysis_df))
print("Patients missing Step-1 risk score:", missing_risk_mask.sum())


### Patients without a Step-1 score

Inspect any unmatched patients before they are excluded from the model.


In [ ]:
analysis_df.loc[
    missing_risk_mask,
    [
        "STUDY_ID",
        "patient_id",
        "study_arm",
        "phq9_baseline",
        "phq9_12m",
    ],
].sort_values(["STUDY_ID", "patient_id"])


In [ ]:
analysis_df[
    [
        "phq9_baseline",
        "risk_score",
        "phq9_12m",
        "outcome_month",
    ]
].describe()


## 6. Define and attach the 13 collaborative-care components

The component indicators represent the manually harmonized intervention-package content.

Control patients correctly have no intervention components. Intervention patients inherit the components assigned to their randomized intervention arm.


In [ ]:
component_cols = [
    "treatment_counseling_was_provided_in_addition_to_medication_(≤ 8 sessions)",
    "treatment_manual_based_psychotherapy",
    "treatment_patient_preference_was_considered_in_treatment_decisions",
    "treatment_automated_follow_up_process_was_used",
    "treatment_is_there_a_relapse_prevention_plan",
    "treatment_were_family_and_friends_involved",
]


In [ ]:
patient_components = (
    filtered_df[
        ["STUDY_ID", "patient_id"] + component_cols
    ]
    .groupby(
        ["STUDY_ID", "patient_id"],
        as_index=False,
    )
    .first()
)

analysis_df = analysis_df.merge(
    patient_components,
    on=["STUDY_ID", "patient_id"],
    how="left",
    validate="one_to_one",
)


## 7. Encode component variables

For the binary components:

- `yes` → 1
- `no` → 0

For `treatment_form_of_monitoring`:

- `no` → 0
- `in_person` → 1
- `by_phone` → 1

The encoding is performed before construction of the numeric design matrices.


In [ ]:
# Encode selected binary components: yes = 1, no = 0

for col in component_cols:
    analysis_df[col] = (
        analysis_df[col]
        .map(
            {
                "no": 0,
                "yes": 1,
            }
        )
    )



In [ ]:
component_missingness = (
    analysis_df[component_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

component_missingness

### Check component missingness

Do not silently interpret unmapped labels as zeros. Any remaining missing values are displayed here before the final complete-case model dataset is created.


In [ ]:
component_missingness = (
    analysis_df[component_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

component_missingness


In [ ]:
for col in component_cols:
    values = analysis_df[col].value_counts(dropna=False).sort_index()
    print(f"\n{col}")
    print(values)


## 8. Construct the final model dataframe

The fitted model requires complete values for:

- `STUDY_ID`;
- `patient_id`;
- `study_arm`;
- approximately 12-month PHQ-9;
- Step-1 `risk_score`;
- all 13 component indicators.

Rows missing any required model variable are excluded at this stage.


In [ ]:
model_columns = [
    "STUDY_ID",
    "patient_id",
    "study_arm",
    "phq9_12m",
    "risk_score",
] + component_cols

model_df = (
    analysis_df[model_columns]
    .dropna()
    .copy()
)

print("Final model patients:", len(model_df))
print("Final model studies:", model_df["STUDY_ID"].nunique())
print(
    "Patients excluded after model requirements:",
    len(analysis_df) - len(model_df),
)


## 9. Build the Step-2 design matrices

The component matrix is

$$
X_C \in \mathbb{R}^{n \times 13}.
$$

The component-by-risk matrix is

$$
X_{C\times R}
=
X_C \odot R.
$$

For patient $i$, each interaction predictor is simply

$$
C_{ik}R_i.
$$

These are predictors, not model predictions.


In [ ]:
X_components = (
    model_df[component_cols]
    .to_numpy(dtype=float)
)

risk = (
    model_df["risk_score"]
    .to_numpy(dtype=float)
)

X_component_risk = (
    X_components
    * risk[:, None]
)

y = (
    model_df["phq9_12m"]
    .to_numpy(dtype=float)
)

study_codes, study_names = pd.factorize(
    model_df["STUDY_ID"]
)

print("X_components:", X_components.shape)
print("risk:", risk.shape)
print("X_component_risk:", X_component_risk.shape)
print("y:", y.shape)


## 10. Rank and package diagnostics

### 10.1 Patient-level $n \times 13$ component matrix

Rank is the number of linearly independent component directions represented by the observed treatment design.

$$
\operatorname{rank}(X_C)
\leq
13.
$$

If the rank is below 13, the likelihood cannot uniquely separate all 13 component-effect directions.


In [ ]:
component_rank = np.linalg.matrix_rank(X_components)

print("Component matrix shape:", X_components.shape)
print("Component rank:", component_rank)
print("Number of components:", len(component_cols))
print(
    "Main-effect rank deficiency:",
    len(component_cols) - component_rank,
)


### 10.2 Unique component packages

Thousands of patients can repeat the same randomized package. Therefore, inspect the number and rank of the **unique component vectors** separately.


In [ ]:
unique_packages = (
    model_df[component_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_package_matrix = unique_packages.to_numpy(dtype=float)
unique_package_rank = np.linalg.matrix_rank(unique_package_matrix)

print("Unique component packages:", len(unique_packages))
print("Rank of unique packages:", unique_package_rank)
print(
    "Package-rank deficiency relative to 13 components:",
    len(component_cols) - unique_package_rank,
)


### 10.3 Which study arms share the same package?

`study_arm` here means the randomized arm label (`control` or `intervention`).

The table below assigns a package ID to each distinct 13-component vector and shows when the same vector occurs in more than one study arm. Control arms are expected to share the all-zero component package.


In [ ]:
package_patterns = (
    model_df[
        ["STUDY_ID", "study_arm"] + component_cols
    ]
    .drop_duplicates()
    .copy()
)

package_patterns["package_id"] = (
    package_patterns
    .groupby(component_cols, dropna=False)
    .ngroup()
)

package_counts = (
    package_patterns
    .groupby("package_id")
    .size()
    .rename("n_study_arms")
    .sort_values(ascending=False)
)

package_counts


In [ ]:
duplicate_package_ids = package_counts[
    package_counts > 1
].index

duplicate_packages = (
    package_patterns.loc[
        package_patterns["package_id"].isin(
            duplicate_package_ids
        )
    ]
    .sort_values(
        ["package_id", "STUDY_ID", "study_arm"]
    )
)

duplicate_packages


### 10.4 Full PATH-style predictor block

The full Step-2 predictor matrix contains:

$$
13 \text{ component columns}
+
1 \text{ risk-score column}
+
13 \text{ component}\times\text{risk columns}
=
27 \text{ columns}.
$$

Its rank is also checked because the interaction expansion may contain additional linear dependencies.


In [ ]:
X_step2 = np.column_stack(
    [
        X_components,
        risk,
        X_component_risk,
    ]
)

step2_rank = np.linalg.matrix_rank(X_step2)

print("Step-2 design shape:", X_step2.shape)
print("Step-2 design rank:", step2_rank)
print(
    "Step-2 rank deficiency:",
    X_step2.shape[1] - step2_rank,
)


## 11. Bayesian PATH-CNMA model

The fitted mean for patient $i$ is

$$
\mu_i
=
\alpha_{\mathrm{study}(i)}
+
\lambda R_i
+
X_{C,i}\beta
+
X_{C\times R,i}\delta.
$$

The model estimates:

- one study-specific intercept for each retained study;
- one prognostic-score main effect;
- 13 component main effects;
- 13 component-by-risk interactions;
- one residual standard deviation.

The priors and sampling settings below are unchanged from the working version.


In [ ]:
coords = {
    "study": study_names.tolist(),
    "component": component_cols,
    "observation": np.arange(len(model_df)),
}


In [ ]:
with pm.Model(coords=coords) as path_cnma_model:

    # Study-specific intercepts
    alpha_study = pm.Normal(
        "alpha_study",
        mu=0,
        sigma=5,
        dims="study",
    )

    # Step-1 prognostic risk-score main effect
    lambda_risk = pm.Normal(
        "lambda_risk",
        mu=0,
        sigma=1,
    )

    # Average component main effects
    beta_component = pm.Normal(
        "beta_component",
        mu=0,
        sigma=1,
        dims="component",
    )

    # Component × prognostic-score interactions
    delta_component_risk = pm.Normal(
        "delta_component_risk",
        mu=0,
        sigma=1,
        dims="component",
    )

    mu = (
        alpha_study[study_codes]
        + lambda_risk * risk
        + pm.math.dot(
            X_components,
            beta_component,
        )
        + pm.math.dot(
            X_component_risk,
            delta_component_risk,
        )
    )

    sigma = pm.HalfNormal(
        "sigma",
        sigma=5,
    )

    y_obs = pm.Normal(
        "y_obs",
        mu=mu,
        sigma=sigma,
        observed=y,
        dims="observation",
    )

    trace_path_cnma = pm.sample(
        draws=4000,
        tune=4000,
        chains=4,
        target_accept=0.99,
        random_seed=42,
    )


## 12. Posterior summary

The posterior summary reports:

- study-specific intercepts;
- the Step-1 prognostic-score coefficient;
- component main effects;
- component-by-risk interactions;
- residual standard deviation.

The 95% HDIs describe posterior uncertainty. `r_hat`, ESS, and MCSE assess MCMC sampling quality, but good MCMC diagnostics do **not** remove structural rank limitations in the treatment design.


In [ ]:
posterior_summary = az.summary(
    trace_path_cnma,
    var_names=[
        "alpha_study",
        "lambda_risk",
        "beta_component",
        "delta_component_risk",
        "sigma",
    ],
    hdi_prob=0.95,
)

posterior_summary


## 13. Interpreting component effects

For component $k$, the model-implied component effect at prognostic score $R$ is

$$
\tau_k(R)
=
\beta_k
+
\delta_k R.
$$

Therefore:

- $\beta_k$ is the component effect when $R=0$;
- $\delta_k$ describes how that component effect changes as the prognostic score increases.

Because the Step-1 risk score is currently **not centered**, $R=0$ may not represent a typical patient and may even lie outside the observed score range. For that reason, component main effects should be interpreted together with the interaction term and evaluated at clinically relevant observed risk-score values.

The risk score is used as a single prognostic summary. Baseline PHQ-9, age, sex, and GAD-7 are therefore **not added again as separate Step-2 predictors** in this PATH-style model.

Finally, finite Bayesian estimates and excellent MCMC convergence do not imply that every individual component coefficient is structurally identifiable. The rank diagnostics above describe the information supplied by the observed treatment-package design.


### Observed Step-1 risk-score range in the fitted cohort


In [ ]:
pd.Series(
    risk,
    name="risk_score",
).describe()


In [ ]:
import json

posterior_summary = az.summary(
    trace_path_cnma,
    var_names=[
        "alpha_study",
        "lambda_risk",
        "beta_component",
        "delta_component_risk",
        "sigma",
    ],
    hdi_prob=0.95,
)

posterior_summary.to_csv("path_cnma_coefficients.csv")

export_payload = {
    "coefficients": (
        posterior_summary
        .reset_index()
        .rename(columns={"index": "parameter"})
        .to_dict(orient="records")
    ),
    "component_order": component_cols,
    "formula": (
        "predicted_phq9_12m = alpha_study[study] + lambda_risk * risk_score "
        "+ sum_k(beta_component[k] * component[k]) "
        "+ sum_k(delta_component_risk[k] * component[k] * risk_score)"
    ),
}

with open("path_cnma_export.json", "w") as f:
    json.dump(export_payload, f, indent=2)

print("Saved path_cnma_coefficients.csv and path_cnma_export.json")

In [ ]:
missing_component_rows = analysis_df[
    analysis_df[component_cols].isna().any(axis=1)
]

print("Patients with any missing selected component:", len(missing_component_rows))

missing_component_rows.groupby(
    ["STUDY_ID", "study_arm"]
).size()

In [ ]:
from pathlib import Path
import json
import platform

# --------------------------------------------------
# Export 6-component PATH-CNMA model
# --------------------------------------------------

EXPORT_DIR = Path("exports/path_cnma_6components_v1")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

POSTERIOR_VARS = [
    "alpha_study",
    "lambda_risk",
    "beta_component",
    "delta_component_risk",
    "sigma",
]

# Make sure this is really the 6-component fit
assert len(component_cols) == 6, (
    f"Expected 6 components, found {len(component_cols)}"
)

assert (
    trace_path_cnma.posterior["beta_component"].sizes["component"] == 6
), "Posterior does not contain 6 component coefficients"

assert (
    trace_path_cnma.posterior["delta_component_risk"].sizes["component"] == 6
), "Posterior does not contain 6 component-risk interactions"

# Recompute packages using only the 6 selected components
unique_packages = (
    model_df[component_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

# --------------------------------------------------
# Export posterior
# Excludes individual observed patient outcomes
# --------------------------------------------------

export_idata = az.InferenceData(
    posterior=trace_path_cnma.posterior[POSTERIOR_VARS],
    sample_stats=trace_path_cnma.sample_stats,
)

export_idata.to_netcdf(
    EXPORT_DIR / "path_cnma_6components_posterior.nc"
)


# --------------------------------------------------
# JSON helper
# --------------------------------------------------

def json_value(value):
    return value.item() if isinstance(value, np.generic) else value


# --------------------------------------------------
# Model manifest
# --------------------------------------------------

manifest = {
    "model_name": "PATH-CNMA Step 2 - 6 Component Model",
    "artifact_version": "1.0.0",

    "outcome": "phq9_12m",

    "risk_score": {
        "source": "Step-1 model",
        "transformation": "none",
        "training_min": float(risk.min()),
        "training_max": float(risk.max()),
    },

    "n_components": 6,

    "component_order": component_cols,

    "study_order": [
        json_value(value)
        for value in study_names.tolist()
    ],

    "allowed_component_packages": (
        unique_packages
        .astype(int)
        .to_dict(orient="records")
    ),

    "posterior": {
        "chains": int(
            trace_path_cnma.posterior.sizes["chain"]
        ),
        "draws_per_chain": int(
            trace_path_cnma.posterior.sizes["draw"]
        ),
        "variables": POSTERIOR_VARS,
        "n_component_main_effects": 6,
        "n_component_risk_interactions": 6,
    },

    "formula": (
        "mu = alpha_study[study] "
        "+ lambda_risk*risk_score "
        "+ sum_k_for_6_components("
        "component[k] * "
        "(beta_component[k] "
        "+ delta_component_risk[k]*risk_score))"
    ),

    "component_effect_formula": (
        "effect_k = beta_component[k] "
        "+ delta_component_risk[k]*risk_score"
    ),

    "versions": {
        "python": platform.python_version(),
        "pymc": pm.__version__,
        "arviz": az.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

with open(
    EXPORT_DIR / "path_cnma_6components_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )

print("6-component PATH-CNMA model exported successfully.")
print(f"Export directory: {EXPORT_DIR.resolve()}")
print(f"Number of components: {len(component_cols)}")
print(f"Number of allowed packages: {len(unique_packages)}")

In [ ]:
# --------------------------------------------------
# Rank checks for the 6-component Step-2 model
# --------------------------------------------------

# 1. Rank of the 6 component main-effect columns
component_rank = np.linalg.matrix_rank(X_components)

print("Component matrix shape:", X_components.shape)
print("Component rank:", component_rank)
print("Number of component columns:", X_components.shape[1])
print(
    "Component rank deficiency:",
    X_components.shape[1] - component_rank,
)

# 2. Rank of the full Step-2 predictor matrix:
#    6 components + 1 risk score + 6 component×risk interactions
X_step2 = np.column_stack(
    [
        X_components,
        risk,
        X_component_risk,
    ]
)

step2_rank = np.linalg.matrix_rank(X_step2)

print("\nFull Step-2 design shape:", X_step2.shape)
print("Full Step-2 rank:", step2_rank)
print("Number of predictor columns:", X_step2.shape[1])
print(
    "Full Step-2 rank deficiency:",
    X_step2.shape[1] - step2_rank,
)